In [21]:
import pandas as pd
from sqlalchemy import create_engine,text
import urllib
from scipy import stats


In [5]:
connection_string = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=Guviproject;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

engine = create_engine(
    "mssql+pyodbc:///?odbc_connect=" + urllib.parse.quote_plus(connection_string)
)
print("Database connected")

Database connected


<h3>T-Test — Delivery & Customer Satisfaction
</h3>

In [6]:
query = """
SELECT
    o.order_id,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,
    r.review_score,
    CASE
        WHEN (o.order_delivered_customer_date)
             > (o.order_estimated_delivery_date)
        THEN 'Late'
        ELSE 'On time'
    END AS delivery_status
FROM olist_orders_dataset AS o
INNER JOIN olist_order_reviews_dataset AS r
    ON o.order_id = r.order_id
WHERE o.order_delivered_customer_date IS NOT NULL
  AND o.order_estimated_delivery_date IS NOT NULL
  AND r.review_score IS NOT NULL;
"""

CustStatisfactionDF = pd.read_sql(query, engine)

CustStatisfactionDF.head()

,order_id,order_delivered_customer_date,order_estimated_delivery_date,review_score,delivery_status
0,9c987217fb96ab86ca15a379d251595c,2017-10-08 13:12:29,2017-10-19,3,On time
1,de38b62136dcb5ff40d44d807c460a8a,2018-07-27 20:02:30,2018-08-13,5,On time
2,8c9bf4b39dc933aa382d9a6e5b846b9d,2017-05-30 15:32:54,2017-06-02,5,On time
3,12d34cd89b0be0896cfecdbd41f5e952,2017-08-22 19:11:00,2017-08-31,4,On time
4,a536641274ad7eca34443ef56220524d,2018-07-14 11:07:42,2018-07-25,5,On time


In [8]:
late_scores = CustStatisfactionDF.loc[
    CustStatisfactionDF["delivery_status"] == "Late", "review_score"
]
on_time_scores = CustStatisfactionDF.loc[
    CustStatisfactionDF["delivery_status"] == "On time", "review_score"
]

In [9]:
if late_scores.empty or on_time_scores.empty:
    raise ValueError(
        "The test requires at least one valid review in both delivery groups."
    )


In [22]:
result = stats.ttest_ind(
    late_scores,
    on_time_scores,
    equal_var=False,
    alternative="less",
)

# Display group summaries and test results.
print("Delivery & Customer Satisfaction")
print("Independent two-sample Welch's t-test\n")
print(f"Valid reviewed orders: {len(CustStatisfactionDF)}")
print(f"Late orders:           {len(late_scores)}")
print(f"On-time orders:        {len(on_time_scores)}")
print(f"Mean score, late:      {late_scores.mean():.3f}")
print(f"Mean score, on time:   {on_time_scores.mean():.3f}")
print(f"Mean difference:       {late_scores.mean() - on_time_scores.mean():.3f}")
print(f"t-statistic:           {result.statistic:.4f}")
print(f"One-sided p-value:     {result.pvalue}")

Delivery & Customer Satisfaction
Independent two-sample Welch's t-test

Valid reviewed orders: 96359
Late orders:           7701
On-time orders:        88658
Mean score, late:      2.567
Mean score, on time:   4.294
Mean difference:       -1.727
t-statistic:           -89.5508
One-sided p-value:     0.0


In [12]:
alpha = 0.05
if result.pvalue < alpha:
    print(
        "\nConclusion: At the 5% significance level, there is evidence that "
        "late orders receive lower review scores on average."
    )
else:
    print(
        "\nConclusion: At the 5% significance level, there is insufficient "
        "evidence that late orders receive lower review scores on average."
    )


Conclusion: At the 5% significance level, there is evidence that late orders receive lower review scores on average.


<h3>ANOVA — Product Category & Spending</h3>

In [16]:
query = """
SELECT 
    p.product_category_name,
    oi.order_id,
    SUM(oi.price) AS order_value
FROM 
    olist_order_items_dataset AS oi(nolock)
JOIN 
    olist_products_dataset AS p(nolock)
ON oi.product_id = p.product_id
GROUP BY 
    p.product_category_name, 
    oi.order_id;
"""

ProductSpending = pd.read_sql(query, engine)

ProductSpending.head()

,product_category_name,order_id,order_value
0,moveis_decoracao,000229ec398224ef6ca0657da4fc703e,199.00
1,ferramentas_jardim,000576fe39319847cbb9d288c5617fa6,810.00
2,livros_tecnicos,0005f50442cb953dcd1d21e1fb923495,53.99
3,consoles_games,0009c9a17f916a706d71784483a5d643,639.00
4,cool_stuff,0010b2e5201cc5f1ae7e9c6cc8f5bd00,48.90


In [24]:
ProductSpendingdf = ProductSpending.dropna(subset=["product_category_name", "order_value"]).copy()
ProductSpendingdf["order_value"] = pd.to_numeric(ProductSpendingdf["order_value"], errors="coerce")
ProductSpendingdf = ProductSpendingdf.dropna(subset=["order_value"])

In [26]:
counts = ProductSpendingdf.groupby("product_category_name")["order_value"].size()
print(counts.sort_values())

product_category_name
seguros_e_servicos                  2
fashion_roupa_infanto_juvenil       8
cds_dvds_musicais                  12
la_cuisine                         13
artes_e_artesanato                 23
                                 ... 
moveis_decoracao                 6449
informatica_acessorios           6689
esporte_lazer                    7720
beleza_saude                     8836
cama_mesa_banho                  9417
Name: order_value, Length: 71, dtype: int64


In [33]:
valid_categories = counts[counts >= 2].index
groups = [
    group["order_value"].to_numpy()
    for _, group in ProductSpendingdf[ProductSpendingdf["product_category_name"].isin(valid_categories)]
        .groupby("product_category_name")
]


In [34]:
if len(groups) < 2:
    print("ANOVA cannot be run: fewer than two categories have at least two observations.")
else:
    f_stat, p_value = stats.f_oneway(*groups)
    print(f"F-statistic: {f_stat:.4f}")
    print(f"p-value: {p_value:.6g}")

    if p_value < 0.05:
        print("Conclusion: There is a statistically significant difference in average order value across product categories.")
    else:
        print("Conclusion: No statistically significant difference found in average order value across product categories.")

F-statistic: 158.5899
p-value: 0
Conclusion: There is a statistically significant difference in average order value across product categories.
